# 📚 Конспект: Цикли, Словники та Comprehensions

> Урок 7 — повний конспект з теорією, патернами алгоритмічного мислення,
> прикладами та типовими помилками.

---

## 📋 Зміст

| # | Тема | Ключові концепції |
|---|------|-------------------|
| 1 | Ментальна модель циклів | `for` vs `while`, `iter()`, `next()` |
| 2 | `range`, зрізи, `enumerate` | генератор діапазону, C-style помилка |
| 3 | Ітерація по словниках | `.keys()`, `.values()`, `.items()` |
| 4 | Безпечна робота зі словниками | `.get()`, `.setdefault()`, `defaultdict` |
| 5 | Патерн: Iteration | конвеєрна стрічка, пошук |
| 6 | Патерн: Mapping | фарбувальна камера, трансформація |
| 7 | Патерн: Aggregation | снігова куля, акумулятор |
| 8 | Патерн: Counting | поштові скриньки, `Counter` |
| 9 | Патерн: Grouping | `defaultdict(list)` |
| 10 | List Comprehensions | ментальна модель, синтаксис |
| 11 | Потік даних (Pipeline) | Data Flow, декомпозиція |
| 12 | Типові помилки | застереження, виправлення |
| 13 | Шпаргалка | швидкий довідник |
| 14 | Задачі | самостійна робота |

---

## 🧠 Як мислить програміст

Програмісти бачать за сирими даними їхню **кінцеву форму** і розуміють,
через які трансформації вони мають пройти.
Вони мислять не рядками коду, а **патернами обробки даних**.

Коли програміст бачить нову задачу, він ставить чотири питання:

```
1. ITERATION  — Як я буду перебирати (ітерувати) ці дані?
2. MAPPING    — Як мені перетворити (map) їх у потрібний формат?
3. COUNTING   — Чи потрібно мені підрахувати якісь частоти або збіги?
4. AGGREGATION— Як зібрати (reduce) все це в кінцеву відповідь?
```

Увесь цей урок — про ці чотири патерни та інструменти Python для їх реалізації.

---

## ⚙️ Частина 1: Ментальна модель циклів

### `for` vs `while` — як вони влаштовані зсередини

**`while`** — керується **умовою**.
Перевіряє булевий вираз і повторює блок доки він `True`.
Ви самі керуєте станом (лічильником).

**`for`** — керується **колекцією**.
Python сам викликає `iter()` на послідовності, щоб отримати «ітератор»,
а потім `next()` щоб брати елементи один за одним — до сигналу `StopIteration`.

```
for letter in ['c', 'a', 't']:     ← Python сам управляє індексом
    print(letter)

Під капотом:
  _iter = iter(['c', 'a', 't'])    ← отримуємо ітератор
  letter = next(_iter)  → 'c'
  letter = next(_iter)  → 'a'
  letter = next(_iter)  → 't'
  next(_iter)           → StopIteration → цикл завершений
```

**Коли що використовувати:**

| Цикл | Використовуйте коли... | Приклад |
|------|------------------------|---------|
| `for` | Є скінченна відома колекція | Список, рядок, `range()` |
| `while` | Повторюємо до настання події | Очікування вводу, ігровий цикл |

In [ ]:
# Частина 1.1: Та сама задача — два цикли
letters = ['c', 'a', 't']

print("for (пітонічно):")
for letter in letters:            # Python сам бере кожен елемент
    print(f"  {letter}")

print()
print("while (ручне управління):")
i = 0
while i < len(letters):           # потрібно самому стежити за i
    print(f"  {letters[i]}")
    i += 1                        # якщо забути — нескінченний цикл!

print()
print("Результат однаковий, але for — чистіший і безпечніший.")

In [ ]:
# Частина 1.2: Під капотом — iter() та next()
data = [10, 20, 30]

# Що Python робить «за лаштунками» при for x in data:
_iter = iter(data)                 # крок 1: отримуємо ітератор
print(f"iter(data) = {_iter}")
print()

print(f"next() → {next(_iter)}")  # крок 2: перший елемент
print(f"next() → {next(_iter)}")  # крок 3: другий
print(f"next() → {next(_iter)}")  # крок 4: третій


In [ ]:
# Частина 1.2: Під капотом — iter() та next()
data = [10, 20, 30]

# Що Python робить «за лаштунками» при for x in data:
_iter = iter(data)                 # крок 1: отримуємо ітератор
print(f"iter(data) = {_iter}")
print()

print(f"next() → {next(_iter)}")  # крок 2: перший елемент
print(f"next() → {next(_iter)}")  # крок 3: другий
print(f"next() → {next(_iter)}")  # крок 4: третій

try:
    print(f"next() → {next(_iter)}")  # більше елементів немає
except StopIteration:
    print("next() → StopIteration! ← for отримує цей сигнал і зупиняється")
print()

# Будь-який об'єкт з __iter__ та __next__ можна використовувати у for
print("Що можна ітерувати:")
iterables = [
    ([1, 2, 3], "list"),
    ((1, 2, 3), "tuple"),
    ({1, 2, 3}, "set"),
    ({"a": 1}, "dict"),
    ("hello",  "str"),
    (range(3), "range"),
]
for obj, name in iterables:
    print(f"  {name:<8}: iter({name}) → {type(iter(obj)).__name__}")

---

## 🔢 Частина 2: `range`, зрізи та `enumerate`

`range(start, stop, step)` генерує послідовність чисел **на льоту** — без створення списку в пам'яті.
Це важливо при роботі з мільйонами чисел.

```python
range(5)          # 0, 1, 2, 3, 4
range(2, 10)      # 2, 3, 4, 5, 6, 7, 8, 9
range(0, 10, 2)   # 0, 2, 4, 6, 8
range(10, 0, -1)  # 10, 9, 8, ..., 1
```

### ⚠️ Застереження 1: C-style індексація

Початківці з інших мов пишуть `for i in range(len(items))` — це **непітонічно**.

```python
# ❌ Непітонічно (C-style)
for i in range(len(items)):
    print(items[i])

# ✅ Пітонічно
for item in items:
    print(item)

# ✅ Якщо потрібен і індекс — enumerate()
for i, item in enumerate(items):
    print(i, item)
```

In [ ]:
# Частина 2.1: range — різні форми
print("range(5)          :", list(range(5)))
print("range(2, 8)       :", list(range(2, 8)))
print("range(0, 10, 2)   :", list(range(0, 10, 2)))
print("range(10, 0, -1)  :", list(range(10, 0, -1)))
print("range(0, -5, -1)  :", list(range(0, -5, -1)))
print()

# range економить пам'ять
import sys
big_range = range(1_000_000)
big_list  = list(range(1_000_000))
print(f"range(1_000_000): {sys.getsizeof(big_range):>10} байт")
print(f"list(range(...)) : {sys.getsizeof(big_list):>10} байт")
print(f"Різниця: x{sys.getsizeof(big_list)//sys.getsizeof(big_range):.0f} пам'яті")

In [ ]:
# Частина 2.2: enumerate — коли потрібен і елемент, і індекс
fruits = ['яблуко', 'банан', 'апельсин', 'ківі']

print("❌ C-style (непітонічно):")
for i in range(len(fruits)):
    print(f"  [{i}] {fruits[i]}")
print()

print("✅ enumerate (пітонічно):")
for i, fruit in enumerate(fruits):
    print(f"  [{i}] {fruit}")
print()

print("✅ enumerate зі стартом з 1:")
for rank, fruit in enumerate(fruits, start=1):
    print(f"  {rank}. {fruit}")
print()

# Ітерація по зрізу
word = "Python"
print(f"word = {word!r}")
print(f"word[1:4] = {word[1:4]!r}")
print("Ітерація по word[1:4]:")
for char in word[1:4]:
    print(f"  {char!r}")

---

## 📖 Частина 3: Ітерація по словниках

Словники зберігають дані у парах `ключ: значення`.
Є три способи ітерувати по них — і кожен дає різний результат.

```
d = {'Alice': 95, 'Bob': 82}

for key in d:              → 'Alice', 'Bob'          (тільки ключі)
for key in d.keys():       → 'Alice', 'Bob'          (явно — ключі)
for val in d.values():     → 95, 82                  (тільки значення)
for k, v in d.items():     → ('Alice', 95), ('Bob', 82)  (пари)
```

### ⚠️ Застереження 2: Забули `.items()` при розпакуванні

```python
d = {'a': 1, 'b': 2}

# ❌ Помилка: for k, v in d  — розпаковує рядок ключа, не пару!
for key, value in d:        # ValueError!
    print(key, value)

# ✅ Правильно:
for key, value in d.items():
    print(key, value)
```

In [ ]:
# Частина 3.1: Три способи ітерувати словник
student_grades = {'Alice': 95, 'Bob': 82, 'Charlie': 88, 'Diana': 76}

print("for key in d (тільки ключі — за замовчуванням):")
for key in student_grades:
    print(f"  {key}")
print()

print("for val in d.values() (тільки значення):")
for score in student_grades.values():
    print(f"  {score}")
print()

print("for k, v in d.items() (пари — найчастіше потрібно):")
for name, score in student_grades.items():
    grade = 'A' if score >= 90 else 'B' if score >= 80 else 'C'
    print(f"  {name:<10}: {score}  ({grade})")

In [ ]:
# Частина 3.2: Демонстрація помилки з .items()
d = {'a': 1, 'b': 2}

print("=== Що відбувається при 'for k, v in d' (БЕЗ .items()) ===")
print(f"При for key in d ми отримуємо рядки: {list(d)}")
print(f"Розпакування рядка 'ab' → (a, b) — це символи, не пари!")
print()

try:
    for key, value in d:   # d дає 'a', 'b' — рядки по 1 символу
        print(key, value)
except ValueError as e:
    print(f"ValueError: {e}")
    print("  Рядок 'a' містить 1 символ, а ми очікуємо 2 значення!")
print()

print("✅ Правильно — через .items():")
for key, value in d.items():
    print(f"  {key} → {value}")

---

## 🔒 Частина 4: Безпечна робота зі словниками

Звернення до відсутнього ключа — `KeyError`. Є три рівні захисту:

| Метод | Поведінка | Коли використовувати |
|-------|-----------|----------------------|
| `d[key]` | `KeyError` якщо немає | Ключ гарантовано є |
| `d.get(key, default)` | Повертає `default` якщо немає | Читання з запасним значенням |
| `d.setdefault(key, default)` | Створює ключ якщо немає, повертає значення | Ініціалізація при першому зверненні |
| `defaultdict(factory)` | Автоматично створює значення | Накопичення (групування, підрахунок) |

In [ ]:
# Частина 4.1: .get() — безпечне читання
grades = {'Alice': 95, 'Bob': 82}

# Небезпечно
try:
    score = grades['David']   # KeyError!
except KeyError as e:
    print(f"grades['David'] → KeyError: {e}")

# Безпечно
score = grades.get('David', 0)     # ключа немає → повертає 0
print(f"grades.get('David', 0)    = {score}")

score = grades.get('Alice', 0)     # ключ є → повертає значення
print(f"grades.get('Alice', 0)    = {score}")

score = grades.get('David')        # default = None
print(f"grades.get('David')       = {score!r}")
print()

# Практичне застосування: підрахунок з get()
text = "hello world"
counts = {}
for char in text:
    counts[char] = counts.get(char, 0) + 1
print(f"Підрахунок символів: {counts}")

In [ ]:
# Частина 4.2: .setdefault() — ініціалізація + доступ
# setdefault(key, default):
#   якщо key є  → повертає існуюче значення (нічого не змінює)
#   якщо key нема → встановлює key=default ТА повертає default

# Задача: згрупувати слова за першою літерою
words = ['apple', 'avocado', 'banana', 'blueberry', 'cherry', 'apricot']
index = {}

for word in words:
    first = word[0]
    # Якщо ключа 'a' немає → створює {'a': []}
    # Якщо є → просто повертає існуючий список
    index.setdefault(first, []).append(word)

print("Групування за першою літерою:")
for letter, group in sorted(index.items()):
    print(f"  '{letter}': {group}")
print()

# Покрокова демонстрація
print("Покроково:")
d = {}
for word in words[:4]:
    key = word[0]
    before = dict(d)
    d.setdefault(key, []).append(word)
    print(f"  setdefault('{key}', []).append('{word}') → {d}")

In [ ]:
# Частина 4.3: defaultdict — автоматичне створення значень
from collections import defaultdict

# defaultdict(factory) викликає factory() для будь-якого нового ключа
# int()   → 0
# list()  → []
# set()   → set()
# str()   → ''

print("=== defaultdict(int) — підрахунок ===")
text = "banana"
letter_counts = defaultdict(int)   # новий ключ → 0 автоматично

for char in text:
    letter_counts[char] += 1       # ніякого KeyError!

print(f"  text: {text!r}")
print(f"  результат: {dict(letter_counts)}")
print()

print("=== defaultdict(list) — групування ===")
purchases = [('Food', 15), ('Transport', 2), ('Food', 30), ('Books', 10), ('Transport', 8)]
grouped = defaultdict(list)        # новий ключ → [] автоматично

for category, amount in purchases:
    grouped[category].append(amount)

for cat, amounts in grouped.items():
    print(f"  {cat:<12}: {amounts}  (сума: {sum(amounts)})")
print()

print("=== Порівняння трьох підходів ===")
print("  dict + if   : if key not in d: d[key] = 0; d[key] += 1")
print("  dict + .get(): d[key] = d.get(key, 0) + 1")
print("  defaultdict  : d[key] += 1  ← найчистіше")

---

## 🏭 Частина 5: Патерн — Iteration (Ітерація)

**Ментальна модель: «Конвеєрна стрічка»**

```
Дані рухаються по стрічці → ви зупиняєте або оглядаєте кожен об'єкт по черзі.
Результат попередньої ітерації → відправна точка для наступної.
```

**Суть мислення:** Перш ніж щось зробити з даними, знайти спосіб пройтись по них
крок за кроком, не загубивши жодного елемента.

**Використання:**
- Пошук елемента (Search)
- Перевірка умов (Validation)
- Прохід по всіх елементах для дії

In [ ]:
# Патерн 5: Iteration — різні форми пошуку
names = ['Alice', 'Bob', 'Charlie', 'David', 'Eve']

print("=== Лінійний пошук ===")
target = 'Charlie'
found = False
for name in names:              # конвеєр
    if name == target:          # огляд
        found = True
        break                   # зупинка конвеєра
print(f"  '{target}' знайдено: {found}")
print()

print("=== Пошук першого що задовольняє умову ===")
numbers = [3, 7, 2, 9, 1, 5, 8]
first_even = None
for n in numbers:
    if n % 2 == 0:
        first_even = n
        break
print(f"  Перше парне в {numbers}: {first_even}")
print()

print("=== Пітонічний пошук через next() + генератор ===")
first_even_v2 = next((n for n in numbers if n % 2 == 0), None)
print(f"  next(n for n if even): {first_even_v2}")
print()

print("=== Перевірка всіх / будь-якого ===")
scores = [85, 92, 78, 95, 88]
print(f"  scores = {scores}")
print(f"  all(s >= 70 for s in scores) = {all(s >= 70 for s in scores)}")
print(f"  any(s >= 90 for s in scores) = {any(s >= 90 for s in scores)}")
print(f"  all(s >= 90 for s in scores) = {all(s >= 90 for s in scores)}")

---

## 🎨 Частина 6: Патерн — Mapping (Трансформація)

**Ментальна модель: «Фарбувальна камера»**

```
На конвеєр надходять сирі деталі
→ камера автоматично фарбує кожну
→ на виході ті самі деталі, але в новому стані

Кількість елементів НЕ змінюється: len(input) == len(output)
```

**Суть мислення:** Як мені змінити кожен елемент (один до одного),
щоб отримати новий набір для подальшої роботи?

**Покрокова логіка:**
1. Взяти сирий елемент
2. Застосувати функцію перетворення
3. Зберегти результат у нову колекцію

In [ ]:
# Патерн 6: Mapping — трансформація форми даних

print("=== Трансформація типів ===")
raw_inputs = ['1', '5', '28', '131', '42']     # рядки з файлу/вводу
integers   = [int(x) for x in raw_inputs]      # перетворюємо в числа
print(f"  Вхід  : {raw_inputs}")
print(f"  Вихід : {integers}")
print(f"  len(in) == len(out): {len(raw_inputs) == len(integers)}")
print()

print("=== Нормалізація тексту ===")
raw_names = ['  alice ', 'BOB', 'CHARLIE  ', ' Diana']
clean = [name.strip().title() for name in raw_names]
print(f"  Вхід  : {raw_names}")
print(f"  Вихід : {clean}")
print()

print("=== Обчислення нового поля ===")
prices_usd = [10.5, 25.0, 3.99, 150.0]
rate       = 40.5
prices_uah = [round(p * rate, 2) for p in prices_usd]
print(f"  USD : {prices_usd}")
print(f"  UAH : {prices_uah}  (курс {rate})")
print()

print("=== map() — функціональна альтернатива ===")
# map(func, iterable) — ліниво застосовує func до кожного елемента
integers_v2 = list(map(int, raw_inputs))
upper_v2    = list(map(str.upper, ['hello', 'world']))
print(f"  map(int, ...)   = {integers_v2}")
print(f"  map(str.upper, ...) = {upper_v2}")
print()
print("Comprehension читабельніший, map() — старший функціональний стиль.")

---

## ❄️ Частина 7: Патерн — Aggregation (Агрегація)

**Ментальна модель: «Снігова куля» (Акумулятор)**

```
Ви створюєте базову змінну-акумулятор (нуль, порожній список...)
→ По мірі того як цикл котиться по даних
→ Змінна «набирає масу», поглинаючи кожен елемент
→ Перехід від БАГАТЬОХ до ОДНОГО
```

**Покрокова логіка:**
1. Ініціалізувати акумулятор (`total = 0`)
2. Отримати наступний елемент
3. Оновити акумулятор (`total += item`)
4. Після циклу — акумулятор містить відповідь

In [ ]:
# Патерн 7: Aggregation — від багатьох до одного

transactions = [10.50, 20.00, 5.25, 40.00, 15.75]
print(f"transactions = {transactions}")
print()

print("=== Ручний акумулятор (ментальна модель) ===")
accumulator = 0
print(f"  Старт: accumulator = {accumulator}")
for t in transactions:
    accumulator += t
    print(f"  += {t:>5} → accumulator = {accumulator:.2f}")
print(f"  Підсумок: {accumulator:.2f}")
print()

print("=== Вбудовані функції-агрегатори ===")
print(f"  sum()   = {sum(transactions):.2f}")
print(f"  min()   = {min(transactions):.2f}")
print(f"  max()   = {max(transactions):.2f}")
print(f"  len()   = {len(transactions)}")
print(f"  avg     = {sum(transactions)/len(transactions):.2f}")
print()

print("=== Складна агрегація: кілька метрик за один прохід ===")
scores = [85, 92, 78, 95, 88, 60, 73]
total = count_above_80 = count_below_70 = 0
maximum = scores[0]

for s in scores:
    total += s
    if s > maximum:  maximum = s
    if s >= 80:      count_above_80 += 1
    if s <  70:      count_below_70 += 1

print(f"  scores        = {scores}")
print(f"  Сума          = {total}")
print(f"  Середнє       = {total/len(scores):.1f}")
print(f"  Максимум      = {maximum}")
print(f"  Вище 80       = {count_above_80}")
print(f"  Нижче 70      = {count_below_70}")

---

## 📬 Частина 8: Патерн — Counting (Підрахунок)

**Ментальна модель: «Поштові скриньки»**

```
Беремо кожен елемент
→ Дивимось на його значення (адресу)
→ Кладемо у відповідну скриньку
→ Якщо скринька порожня — створюємо її
→ Рахуємо висоту стопок у кожній скриньці

Перехід від БАГАТЬОХ до КІЛЬКОХ (статистика розподілу)
```

**Застосування:** частотний аналіз тексту, статистика подій, голосування.

In [ ]:
# Патерн 8: Counting — частотний аналіз
from collections import Counter

print("=== Ручний підрахунок (ментальна модель) ===")
raw_text = "banana"
freq = {}
for char in raw_text:
    if char not in freq:
        freq[char] = 0          # нова «скринька»
    freq[char] += 1             # кладемо елемент у скриньку
print(f"  raw_text = {raw_text!r}")
print(f"  ручний  = {freq}")
print()

print("=== Counter — спеціалізований підрахунок ===")
letter_freq = Counter(raw_text)
print(f"  Counter = {letter_freq}")
print(f"  most_common(2) = {letter_freq.most_common(2)}")
print()

print("=== Практика: частотний аналіз слів ===")
text = "the quick brown fox jumps over the lazy dog the fox"
word_freq = Counter(text.split())
print(f"  Текст: {text!r}")
print(f"  Топ-3 слова: {word_freq.most_common(3)}")
print(f"  Унікальних слів: {len(word_freq)}")
print()

print("=== Counter — арифметика ===")
votes_day1 = Counter({'Alice': 10, 'Bob': 7, 'Carol': 5})
votes_day2 = Counter({'Alice': 8,  'Bob': 12, 'Carol': 3})
total_votes = votes_day1 + votes_day2
print(f"  День 1: {dict(votes_day1)}")
print(f"  День 2: {dict(votes_day2)}")
print(f"  Разом : {dict(total_votes)}")
print(f"  Лідер : {total_votes.most_common(1)[0][0]}")

---

## 🗂️ Частина 9: Патерн — Grouping (Групування)

**Розширення підрахунку:** замість рахувати елементи — збираємо повні записи
у списки за певним ключем.

```
Підрахунок:  { 'Food': 3 }             ← скільки разів
Групування:  { 'Food': [15, 30, 12] }  ← які саме значення
```

**Інструмент:** `defaultdict(list)` — словник де кожен новий ключ автоматично отримує `[]`.

In [ ]:
# Патерн 9: Grouping — побудова індексу
from collections import defaultdict

print("=== Групування транзакцій за категорією ===")
purchases = [
    ('Food',      15), ('Transport',  2), ('Food',    30),
    ('Books',     10), ('Food',       12), ('Transport', 8),
    ('Books',     25), ('Transport',   5),
]

# Крок 1: Групування
grouped = defaultdict(list)
for category, amount in purchases:
    grouped[category].append(amount)   # скринька для кожної категорії

print("  Групи:")
for cat, amounts in sorted(grouped.items()):
    print(f"    {cat:<12}: {amounts}")
print()

# Крок 2: Агрегація по групах
print("  Статистика по групах:")
print(f"  {'Категорія':<12} {'Кількість':>10} {'Сума':>8} {'Середнє':>10}")
print("  " + "-" * 44)
for cat, amounts in sorted(grouped.items()):
    print(f"  {cat:<12} {len(amounts):>10} {sum(amounts):>8} "
          f"{sum(amounts)/len(amounts):>10.1f}")
print()

print("=== Групування слів за довжиною ===")
words = ['cat', 'dog', 'elephant', 'ant', 'bee', 'crocodile', 'fox', 'bear']
by_length = defaultdict(list)
for word in words:
    by_length[len(word)].append(word)

for length, group in sorted(by_length.items()):
    print(f"  {length} символи: {group}")

---

## ⚡ Частина 10: List Comprehensions

**Ментальна модель:** «Приплюснутий `for`-цикл в дужках».

```
[ що_зберегти   for елемент in колекція   if умова ]
  ↑ Transform       ↑ Iterate               ↑ Filter
```

**Переклад:**

```python
# Традиційно (акумулятор):
squares = []
for x in range(5):
    if x % 2 == 0:
        squares.append(x ** 2)

# Comprehension:
squares = [x ** 2 for x in range(5) if x % 2 == 0]
```

**Коли НЕ використовувати:**
Якщо мета — **виконати дію** (print, запис у файл, відправка запиту) — використовуйте звичайний `for`.
Comprehension — тільки коли мета — **побудувати новий список**.

In [ ]:
# Частина 10.1: List Comprehensions — базовий синтаксис

# Форма 1: тільки трансформація (без фільтру)
squares = [x ** 2 for x in range(1, 6)]
print(f"[x**2 for x in range(1,6)] = {squares}")

# Форма 2: тільки фільтрація (без трансформації)
evens = [x for x in range(1, 21) if x % 2 == 0]
print(f"[x for x in 1..20 if even] = {evens}")

# Форма 3: трансформація + фільтрація
even_squares = [x**2 for x in range(1, 11) if x % 2 == 0]
print(f"Квадрати парних 1-10       = {even_squares}")
print()

# Практика: нормалізація тексту
raw = ['  Alice ', 'BOB  ', '', '  Charlie']
clean = [name.strip().title() for name in raw if name.strip()]
print(f"raw   = {raw}")
print(f"clean = {clean}")
print()

# Практика: розплющення (flatten) вкладеного списку
matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
flat   = [n for row in matrix for n in row]
print(f"matrix = {matrix}")
print(f"flat   = {flat}")

In [ ]:
# Частина 10.2: Dict та Set Comprehensions

print("=== Dict Comprehension ===")
# { ключ: значення for елемент in колекція if умова }
words = ['apple', 'banana', 'cherry', 'date']
word_lengths = {w: len(w) for w in words}
print(f"word_lengths = {word_lengths}")

# Інвертувати словник (ключі ↔ значення)
original = {'a': 1, 'b': 2, 'c': 3}
inverted = {v: k for k, v in original.items()}
print(f"original = {original}")
print(f"inverted = {inverted}")
print()

print("=== Set Comprehension ===")
# { вираз for елемент in колекція } — тільки унікальні
text = "hello world"
unique_chars = {c for c in text if c.isalpha()}
print(f"unique chars in {text!r}: {sorted(unique_chars)}")
print()

print("=== Generator Expression (економить пам'ять) ===")
# (вираз for елемент in колекція) — НЕ будує список, обчислює на льоту
import sys
lst_comp  = [x**2 for x in range(10_000)]        # будує список в пам'яті
gen_expr  = (x**2 for x in range(10_000))         # лінивий генератор
print(f"list comprehension: {sys.getsizeof(lst_comp):>8} байт")
print(f"generator expr    : {sys.getsizeof(gen_expr):>8} байт")
print(f"Для sum() генератор кращий: sum(x**2 for x in range(N))")

---

## 🔄 Частина 11: Потік даних (Data Pipeline)

**Ідея:** замість одного великого циклу — розбити задачу на ланцюжок простих кроків.

```
Сирі дані
   │
   ├─ Step 1: Filter   — відкинути непотрібне
   │
   ├─ Step 2: Map      — перетворити форму
   │
   ├─ Step 3: Group    — організувати
   │
   └─ Step 4: Reduce   — агрегувати у відповідь
```

**Ключова відмінність мислення:**

| Підхід | Ідея |
|--------|------|
| **Імперативний** | Явні змінні + зміна стану в циклах. «Зроби це, потім онови це» |
| **Функціональний** | Конвеєр де дані перетікають між функціями. Нові об'єкти, без мутації |

In [ ]:
# Частина 11: Data Pipeline — аналіз лог-файлу продажів

# Сирі дані (імітація CSV: дата, продукт, кількість, ціна)
raw_log = [
    "2024-01-15,Apple,10,2.5",
    "2024-01-15,Banana,ERROR,1.2",   # пошкоджений рядок
    "2024-01-16,Apple,8,2.5",
    "2024-01-16,Cherry,5,4.0",
    "",                               # порожній рядок
    "2024-01-17,Banana,20,1.2",
    "2024-01-17,Apple,15,2.5",
    "2024-01-17,Cherry,ERROR,4.0",   # ще пошкоджений
    "2024-01-18,Banana,12,1.2",
]

print(f"Сирий лог: {len(raw_log)} рядків (з помилками та порожніми)")
print()

# === PIPELINE ===

# Step 1: Filter — тільки валідні рядки (не порожні, кількість числова)
def is_valid(line):
    if not line.strip():
        return False
    parts = line.split(',')
    return len(parts) == 4 and parts[2].isdigit()

valid_lines = [line for line in raw_log if is_valid(line)]
print(f"Step 1 Filter: {len(valid_lines)}/{len(raw_log)} рядків валідні")

# Step 2: Map — перетворити рядки на словники
records = [
    {
        'date':    parts[0],
        'product': parts[1],
        'qty':     int(parts[2]),
        'price':   float(parts[3]),
        'revenue': int(parts[2]) * float(parts[3]),
    }
    for line in valid_lines
    for parts in [line.split(',')]   # трюк: прив'язуємо parts
]
print(f"Step 2 Map: перетворено на {len(records)} записів")

# Step 3: Group — по продукту
from collections import defaultdict
by_product = defaultdict(list)
for r in records:
    by_product[r['product']].append(r)
print(f"Step 3 Group: {len(by_product)} продукти")

# Step 4: Reduce — агрегована статистика
print()
print("=" * 48)
print("  ЗВІТ ПРОДАЖІВ")
print("=" * 48)
print(f"  {'Продукт':<10} {'Кількість':>10} {'Виторг':>10} {'Середнє':>10}")
print("  " + "-" * 44)

for product, recs in sorted(by_product.items()):
    total_qty = sum(r['qty']     for r in recs)
    total_rev = sum(r['revenue'] for r in recs)
    avg_rev   = total_rev / len(recs)
    print(f"  {product:<10} {total_qty:>10} {total_rev:>10.2f} {avg_rev:>10.2f}")

print("  " + "-" * 44)
grand_total = sum(r['revenue'] for r in records)
print(f"  {'РАЗОМ':<10} {sum(r['qty'] for r in records):>10} {grand_total:>10.2f}")
print("=" * 48)

---

## ⚠️ Частина 12: Типові помилки — Застереження

### Застереження 1: C-style індексація
```python
# ❌ Непітонічно
for i in range(len(items)):
    print(items[i])

# ✅ Якщо потрібен тільки елемент
for item in items:
    print(item)

# ✅ Якщо потрібні і елемент, і індекс
for i, item in enumerate(items):
    print(i, item)
```

### Застереження 2: Зміна списку під час ітерації
```python
# ❌ Небезпечно — пропускає елементи!
for item in my_list:
    if condition: my_list.remove(item)

# ✅ Ітеруємо по копії
for item in my_list.copy():
    if condition: my_list.remove(item)

# ✅ Ще краще — comprehension
my_list = [x for x in my_list if not condition]
```

### Застереження 3: Забули `.items()` при розпакуванні
```python
d = {'a': 1, 'b': 2}
for key, value in d:         # ❌ ValueError!
for key, value in d.items(): # ✅
```

In [ ]:
# Частина 12: Демонстрація типових помилок та виправлень

print("=== Застереження 2: Зміна під час ітерації ===")
original = [1, 2, 3, 4, 5, 6]

# ❌ Небезпечно
dangerous = original.copy()
for n in dangerous:
    if n % 2 == 0:
        dangerous.remove(n)   # зсуває індекси!
print(f"Небезпечно (пропуски): {dangerous}  ← 4 залишилась!")

# ✅ Варіант 1: ітерація по копії
safe_v1 = original.copy()
for n in safe_v1.copy():      # safe_v1.copy() — копія
    if n % 2 == 0:
        safe_v1.remove(n)
print(f"Копія (.copy()):       {safe_v1}")

# ✅ Варіант 2: comprehension (найкраще)
safe_v2 = [n for n in original if n % 2 != 0]
print(f"Comprehension:         {safe_v2}")
print()

print("=== Застереження 3: Забули .items() ===")
d = {'apple': 3, 'banana': 7}

print("for k in d — дає ключі:")
for k in d:
    print(f"  k = {k!r}")

try:
    for k, v in d:   # рядок 'apple' → 5 символів, а не 2
        pass
except ValueError as e:
    print(f"for k, v in d → ValueError: {e}")

print("for k, v in d.items() — правильно:")
for k, v in d.items():
    print(f"  {k!r}: {v}")
print()

print("=== Додатково: KeyError vs .get() ===")
grades = {'Alice': 95}
try:
    _ = grades['Bob']        # KeyError
except KeyError as e:
    print(f"grades['Bob'] → KeyError: {e}")
print(f"grades.get('Bob', 0) → {grades.get('Bob', 0)}  ← безпечно")

---

## 📋 Частина 13: Шпаргалка (Quick Reference)

### Цикли
```python
for item in collection:          # пройти по колекції
for i, item in enumerate(lst):   # індекс + елемент
for a, b in zip(lst1, lst2):     # паралельно по двох
for k, v in d.items():           # словник: ключ + значення
for i in range(n):               # 0, 1, ..., n-1
for i in range(a, b, step):      # від a до b-1 з кроком

while condition:                 # поки умова True
    break                        # вихід з циклу
    continue                     # наступна ітерація
```

### Словники
```python
d[key]                           # KeyError якщо немає
d.get(key, default)              # безпечне читання
d.setdefault(key, default)       # читання + ініціалізація
key in d                         # перевірка наявності
d.keys()   d.values()   d.items()

from collections import defaultdict
d = defaultdict(int)             # автоматично 0 для нових ключів
d = defaultdict(list)            # автоматично [] для нових ключів

from collections import Counter
c = Counter(iterable)            # підрахунок елементів
c.most_common(n)                 # топ-n найчастіших
```

### Comprehensions
```python
[expr for x in seq if cond]     # list comprehension
{k: v for k, v in seq}          # dict comprehension
{expr for x in seq}             # set comprehension
(expr for x in seq)             # generator expression
```

### 4 патерни мислення
```python
# ITERATION: next((x for x in seq if cond), default)
# MAPPING:   [transform(x) for x in seq]
# AGGREGATION: sum/min/max/len(seq)
# COUNTING:  Counter(seq)  або  defaultdict(int)
# GROUPING:  defaultdict(list)
```

---

# 📝 Задачі — Самостійна робота

### TODO 1: Частотний аналіз (Counting)

Дано список транзакцій: `(місто, сума)`.
Знайдіть:
1. Кількість транзакцій по кожному місту
2. Загальну суму по кожному місту
3. Місто з найбільшою кількістю транзакцій

In [ ]:
from collections import Counter, defaultdict

transactions = [
    ('Kyiv', 150), ('Lviv', 80),  ('Kyiv', 200),
    ('Odesa', 50), ('Kyiv', 120), ('Lviv', 90),
    ('Odesa', 70), ('Kyiv', 300), ('Odesa', 110),
]

# YOUR CODE HERE
# Підказка 1: Counter(city for city, amount in transactions)
# Підказка 2: defaultdict(int) для сум
# Підказка 3: max(counts, key=counts.get)

# Очікуваний вивід:
# Кількість: Counter({'Kyiv': 4, 'Odesa': 3, 'Lviv': 2})
# Суми: {'Kyiv': 770, 'Lviv': 170, 'Odesa': 230}
# Лідер: Kyiv

### TODO 2: Comprehension Pipeline

Дано список рядків з даними студентів: `"Ім'я,Оцінка"`.
1. Відфільтруйте рядки з помилками (не числова оцінка)
2. Перетворіть на список словників
3. Додайте поле `'grade'`: A≥90, B≥80, C≥70, F<70
4. Виведіть тільки тих хто не F

In [ ]:
raw_data = [
    "Alice,92", "Bob,ERROR", "Charlie,85",
    "Diana,",   "Eve,78",    "Frank,65",
    "Grace,95", "Henry,71",
]

# YOUR CODE HERE
# Step 1: відфільтруйте валідні рядки (score.isdigit())
# Step 2: перетворіть на dict {'name': ..., 'score': int(...)}
# Step 3: додайте 'grade' через умовний вираз або допоміжну функцію
# Step 4: list comprehension тільки не F

# Очікуваний вивід:
# {'name': 'Alice',   'score': 92, 'grade': 'A'}
# {'name': 'Charlie', 'score': 85, 'grade': 'B'}
# {'name': 'Eve',     'score': 78, 'grade': 'C'}
# {'name': 'Grace',   'score': 95, 'grade': 'A'}
# {'name': 'Henry',   'score': 71, 'grade': 'C'}

### TODO 3: Інвертування словника

Є словник `{слово: частота}`. Побудуйте зворотний індекс:
`{частота: [список слів з такою частотою]}`.
Виведіть слова відсортовані за спаданням частоти.

In [ ]:
word_freq = {
    'the': 15, 'a': 12, 'python': 8, 'is': 12,
    'great': 5, 'code': 8, 'language': 5, 'and': 9,
}

# YOUR CODE HERE
# Підказка: defaultdict(list) + for word, freq in word_freq.items()
# Потім відсортуйте: sorted(inverted.items(), key=lambda x: x[0], reverse=True)

# Очікуваний вивід:
# Частота 15: ['the']
# Частота 12: ['a', 'is']
# Частота  9: ['and']
# Частота  8: ['python', 'code']
# Частота  5: ['great', 'language']

### TODO 4: Повний pipeline

Дано список рядків — лог доступу: `"IP,статус_код,байт"`.
1. Відфільтруйте валідні рядки
2. Перетворіть на структури
3. Порахуйте кількість запитів та трафік по IP
4. Знайдіть IP з найбільшим трафіком
5. Порахуйте статуси (200, 404, 500)

In [ ]:
from collections import defaultdict, Counter

access_log = [
    "192.168.1.1,200,1024",
    "10.0.0.1,404,256",
    "192.168.1.1,200,2048",
    "INVALID_LINE",
    "10.0.0.2,500,128",
    "192.168.1.1,200,512",
    "10.0.0.1,200,768",
    "10.0.0.2,404,256",
    "",
    "192.168.1.2,200,4096",
]

# YOUR CODE HERE
# Step 1: is_valid — рядок, що split(',') дає 3 частини і parts[2].isdigit()
# Step 2: перетворіть на {'ip': ..., 'status': int(...), 'bytes': int(...)}
# Step 3: defaultdict(int) для трафіку та Counter для запитів
# Step 4: max(traffic, key=traffic.get)
# Step 5: Counter(r['status'] for r in records)

# Очікуваний вивід:
# Трафік по IP:
#   192.168.1.2: 4096 байт (1 запит)
#   192.168.1.1: 3584 байт (3 запити)
#   10.0.0.1:    1024 байт (2 запити)
#   10.0.0.2:     384 байт (2 запити)
# Лідер трафіку: 192.168.1.2
# Статуси: Counter({200: 6, 404: 2, 500: 1})